In [1]:
pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [61]:
from sqlalchemy import create_engine

username = "root"
password = "1234"
host = "localhost"
port = "3306"
database = "cart2insights"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

In [62]:
import pandas as pd

In [63]:
import pandas as pd

processed_path = r"C:\Users\Administrator\Documents\1st Project\Processed_data"

tables = {
    "customers": "customers_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "order_items": "order_items_clean.csv",
    "order_payments": "order_payments_clean.csv",
    "order_reviews": "order_reviews_clean.csv",
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": "category_translation_clean.csv",
}

for table_name, filename in tables.items():
    df = pd.read_csv(f"{processed_path}\\{filename}")
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {df.shape[0]} rows")

Loaded customers: 99441 rows
Loaded geolocation: 738305 rows
Loaded order_items: 112650 rows
Loaded order_payments: 103886 rows
Loaded order_reviews: 99224 rows
Loaded orders: 99441 rows
Loaded products: 32951 rows
Loaded sellers: 3095 rows
Loaded category_translation: 71 rows


In [66]:
mismatch = pd.read_sql("""
    SELECT DISTINCT p.product_category_name
    FROM products p
    LEFT JOIN category_translation c
    ON p.product_category_name = c.product_category_name
    WHERE c.product_category_name IS NULL;
""", con=engine)
print(mismatch)

Empty DataFrame
Columns: [product_category_name]
Index: []


In [ ]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO category_translation (product_category_name, product_category_name_english)
        VALUES 
            ('unknown', 'unknown'),
            ('pc_gamer', 'pc_gamer'),
            ('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_and_food_preparation')
    """))
    conn.commit()

Block 4 done


In [67]:
check = pd.read_sql("SELECT product_category_name, COUNT(*) as cnt FROM category_translation GROUP BY product_category_name HAVING cnt > 1;", con=engine)
print(check)

Empty DataFrame
Columns: [product_category_name, cnt]
Index: []


In [68]:
pk_statements = [
    "ALTER TABLE customers MODIFY customer_id VARCHAR(100);",
    "ALTER TABLE customers ADD PRIMARY KEY (customer_id);",
    "ALTER TABLE orders MODIFY order_id VARCHAR(100);",
    "ALTER TABLE orders ADD PRIMARY KEY (order_id);",
    "ALTER TABLE products MODIFY product_id VARCHAR(100);",
    "ALTER TABLE products ADD PRIMARY KEY (product_id);",
    "ALTER TABLE sellers MODIFY seller_id VARCHAR(100);",
    "ALTER TABLE sellers ADD PRIMARY KEY (seller_id);",
    "ALTER TABLE category_translation MODIFY product_category_name VARCHAR(100);",
    "ALTER TABLE category_translation ADD PRIMARY KEY (product_category_name);",
    "ALTER TABLE order_reviews ADD PRIMARY KEY (review_pk);",
    "ALTER TABLE order_items MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_items ADD PRIMARY KEY (order_id, order_item_id);",
    "ALTER TABLE order_payments MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_payments ADD PRIMARY KEY (order_id, payment_sequential);",
]

with engine.connect() as conn:
    for stmt in pk_statements:
        conn.execute(text(stmt))
        print("Done:", stmt)
    conn.commit()

Done: ALTER TABLE customers MODIFY customer_id VARCHAR(100);
Done: ALTER TABLE customers ADD PRIMARY KEY (customer_id);
Done: ALTER TABLE orders MODIFY order_id VARCHAR(100);
Done: ALTER TABLE orders ADD PRIMARY KEY (order_id);
Done: ALTER TABLE products MODIFY product_id VARCHAR(100);
Done: ALTER TABLE products ADD PRIMARY KEY (product_id);
Done: ALTER TABLE sellers MODIFY seller_id VARCHAR(100);
Done: ALTER TABLE sellers ADD PRIMARY KEY (seller_id);
Done: ALTER TABLE category_translation MODIFY product_category_name VARCHAR(100);
Done: ALTER TABLE category_translation ADD PRIMARY KEY (product_category_name);
Done: ALTER TABLE order_reviews ADD PRIMARY KEY (review_pk);
Done: ALTER TABLE order_items MODIFY order_id VARCHAR(100);
Done: ALTER TABLE order_items ADD PRIMARY KEY (order_id, order_item_id);
Done: ALTER TABLE order_payments MODIFY order_id VARCHAR(100);
Done: ALTER TABLE order_payments ADD PRIMARY KEY (order_id, payment_sequential);


In [69]:
fk_statements = [
    "ALTER TABLE orders MODIFY customer_id VARCHAR(100);",
    "ALTER TABLE orders ADD FOREIGN KEY (customer_id) REFERENCES customers(customer_id);",
    "ALTER TABLE order_items MODIFY product_id VARCHAR(100);",
    "ALTER TABLE order_items ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE order_items ADD FOREIGN KEY (product_id) REFERENCES products(product_id);",
    "ALTER TABLE order_items MODIFY seller_id VARCHAR(100);",
    "ALTER TABLE order_items ADD FOREIGN KEY (seller_id) REFERENCES sellers(seller_id);",
    "ALTER TABLE order_payments ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE order_reviews MODIFY order_id VARCHAR(100);",
    "ALTER TABLE order_reviews ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);",
    "ALTER TABLE products MODIFY product_category_name VARCHAR(100);",
    "ALTER TABLE products ADD FOREIGN KEY (product_category_name) REFERENCES category_translation(product_category_name);",
]

with engine.connect() as conn:
    for stmt in fk_statements:
        conn.execute(text(stmt))
        print("Done:", stmt)
    conn.commit()

Done: ALTER TABLE orders MODIFY customer_id VARCHAR(100);
Done: ALTER TABLE orders ADD FOREIGN KEY (customer_id) REFERENCES customers(customer_id);
Done: ALTER TABLE order_items MODIFY product_id VARCHAR(100);
Done: ALTER TABLE order_items ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);
Done: ALTER TABLE order_items ADD FOREIGN KEY (product_id) REFERENCES products(product_id);
Done: ALTER TABLE order_items MODIFY seller_id VARCHAR(100);
Done: ALTER TABLE order_items ADD FOREIGN KEY (seller_id) REFERENCES sellers(seller_id);
Done: ALTER TABLE order_payments ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);
Done: ALTER TABLE order_reviews MODIFY order_id VARCHAR(100);
Done: ALTER TABLE order_reviews ADD FOREIGN KEY (order_id) REFERENCES orders(order_id);
Done: ALTER TABLE products MODIFY product_category_name VARCHAR(100);
Done: ALTER TABLE products ADD FOREIGN KEY (product_category_name) REFERENCES category_translation(product_category_name);


In [70]:
for table in ["customers", "geolocation", "orders", "order_items", "order_payments",
              "order_reviews", "products", "sellers", "category_translation"]:
    result = pd.read_sql(f"SHOW CREATE TABLE {table};", con=engine)
    print(f"===== {table} =====")
    print(result['Create Table'][0])
    print()

===== customers =====
CREATE TABLE `customers` (
  `customer_id` varchar(100) NOT NULL,
  `customer_unique_id` text,
  `customer_zip_code_prefix` bigint DEFAULT NULL,
  `customer_city` text,
  `customer_state` text,
  PRIMARY KEY (`customer_id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci

===== geolocation =====
CREATE TABLE `geolocation` (
  `geolocation_zip_code_prefix` bigint DEFAULT NULL,
  `geolocation_lat` double DEFAULT NULL,
  `geolocation_lng` double DEFAULT NULL,
  `geolocation_city` text,
  `geolocation_state` text
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci

===== orders =====
CREATE TABLE `orders` (
  `order_id` varchar(100) NOT NULL,
  `customer_id` varchar(100) DEFAULT NULL,
  `order_status` text,
  `order_purchase_timestamp` text,
  `order_approved_at` text,
  `order_delivered_carrier_date` text,
  `order_delivered_customer_date` text,
  `order_estimated_delivery_date` text,
  PRIMARY KEY (`order_id`),
  KEY `customer_id` (